In [ ]:
import streamlit as st
import torch
import torch.nn as nn
import numpy as np

# ==========================================
# 1. Model Mimarisinin Tanımlanması (Yükleme için gerekli)
# ==========================================
class GRUModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=2, output_size=1):
        super(GRUModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        out, _ = self.gru(x, h0)
        out = self.fc(out[:, -1, :])
        return out

# Modeli belleğe yükleme fonksiyonu
@st.cache_resource
def load_model():
    model = GRUModel()
    # Kaydettiğimiz ağırlıkları yüklüyoruz
    model.load_state_dict(torch.load('capacity_futures_gru.pth', weights_only=True))
    model.eval()
    return model

model = load_model()

# ==========================================
# 2. Arayüz Tasarımı ve Dinamik Fiyatlandırma Mantığı
# ==========================================
st.set_page_config(page_title="Capacity Futures Dashboard", layout="wide")
st.title("🚛 Capacity Futures: Atıl Kapasite ve Dinamik Fiyatlandırma Paneli")
st.markdown("Geçmiş depo envanter seviyelerine göre gelecekteki atıl araç kapasitesini tahmin eden karar destek sistemi.")

st.sidebar.header("Simülasyon Parametreleri")
st.sidebar.write("Son 24 saatin ortalama depo doluluk trendini belirleyin:")

# Kullanıcıdan son 24 saati temsil edecek basitleştirilmiş bir trend değeri alıyoruz
trend_secimi = st.sidebar.slider("Güncel Envanter Yoğunluğu (0: Çok Düşük, 1000: Çok Yüksek)", min_value=0, max_value=1000, value=500, step=10)

if st.sidebar.button("Gelecek Saati Tahmin Et ve Fiyatı Belirle"):
    # Simülasyon için kullanıcının seçtiği trende dayalı 24 saatlik sentetik bir dizi oluşturuyoruz
    # Gerçek bir projede bu veri doğrudan veritabanından son 24 saati çekerek yapılır.
    sentetik_veri = np.linspace(trend_secimi - 50, trend_secimi + 50, 24)
    
    # Veriyi modele uygun formata (-1 ile 1 arası) basitçe ölçekliyoruz (MinMaxScaler simülasyonu)
    scaled_input = (sentetik_veri / 1000.0) * 2 - 1 
    
    # PyTorch Tensor formatına dönüşüm (1, 24, 1)
    tensor_input = torch.tensor(scaled_input, dtype=torch.float32).unsqueeze(0).unsqueeze(-1)
    
    # Model Tahmini
    with torch.no_grad():
        prediction_scaled = model(tensor_input).item()
    
    # Tahmini geri orijinal ölçeğe çevirme
    tahmini_envanter = ((prediction_scaled + 1) / 2) * 1000.0
    
    st.subheader("Tahmin Sonuçları")
    col1, col2, col3 = st.columns(3)
    
    col1.metric("Tahmini Depo Envanteri", f"{tahmini_envanter:.0f} Birim")
    
    # Dinamik Fiyatlandırma Algoritması (Kuyruk/Kapasite Teorisi Yaklaşımı)
    # Envanter yüksekse (depo dolu, mallar çıkmayı bekliyor -> kamyon talebi yüksek) -> Fiyat artar
    # Envanter düşükse (depo boş -> kamyonlar atıl bekliyor) -> Fiyat düşerek talep çekilir
    baz_fiyat = 5000 # Kamyon başı baz taşıma fiyatı (TL)
    
    if tahmini_envanter > 700:
        kapasite_durumu = "Kritik (Yüksek Talep)"
        dinamik_fiyat = baz_fiyat * 1.35 # %35 zamlı
        renk = "red"
    elif tahmini_envanter > 400:
        kapasite_durumu = "Normal (Dengeli)"
        dinamik_fiyat = baz_fiyat
        renk = "green"
    else:
        kapasite_durumu = "Atıl (Boş Araçlar Fazla)"
        dinamik_fiyat = baz_fiyat * 0.75 # %25 indirimli
        renk = "blue"
        
    col2.metric("Kapasite Durumu", kapasite_durumu)
    col3.metric("Önerilen Dinamik Taşıma Fiyatı", f"₺{dinamik_fiyat:,.2f}")
    
    st.markdown("---")
    st.markdown(f"**Stratejik Öneri:** Sistem bir sonraki saat için envanter seviyesinin **{tahmini_envanter:.0f}** birim olacağını öngörüyor. Kapasite durumu **<span style='color:{renk}'>{kapasite_durumu}</span>** olduğu için optimal karlılık ve kaynak kullanımı adına sistemin belirlediği taşıma birim fiyatı **₺{dinamik_fiyat:,.2f}** olarak güncellenmiştir.", unsafe_allow_html=True)